In [ ]:
import numpy as np
import pandas as pd
import uproot
from pathlib import Path
from glob import glob
import re

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mutools.plotting import use_style
from mutools.plotting.helpers import mark_axis

use_style('rootlike')

PATH = Path('/Users/mueller/data/spineosc/v1.0.10')
LOCAL = Path('/Users/mueller/Projects/spineosc/detsys')

In [ ]:
def load_histogram(
    rf: uproot.reading.ReadOnlyDirectory,
    key: str
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load a histogram from a ROOT file and return the bin edges and bin
    contents as numpy arrays.

    Parameters
    ----------
    rf : uproot.reading.ReadOnlyDirectory
        The ROOT file object.
    key : str
        The key of the histogram to load.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        A tuple containing the bin edges and bin contents as numpy arrays.
    """
    hist = rf[key].to_numpy()
    bin_edges = hist[1]
    bin_contents = hist[0]
    return bin_edges, bin_contents

In [ ]:
# Build the relevant patterns
detector_pattern = r'(?P<detector>sbnd|icarus)'
run_pattern = r'(?P<run>run\d+)'
systematic_pattern = r'(?P<systematic>.+?)'
variable_pattern = r'(?P<variable>.+)'
pattern = f'{detector_pattern}_{run_pattern}_{systematic_pattern}_{variable_pattern}'

# Construct the list of files to process
files = LOCAL.glob('profit_systematics/*.root')

# Iterate over the files
data = {}
for file in files:
    base_name = file.stem
    
    match = re.match(pattern, base_name)
    if match:

        # Pack the data from the histograms into the dictionary.
        #
        # The run label is capitalized for the data-driven systematics
        # (WireModYZ_sbnd_Run1) but lowercase for the cut-threshold ones
        # (TriggerEmulationP1_icarus_run4), and the XMLs reference both forms
        # as written, so accept either here rather than renaming anything.
        rf = uproot.open(file)
        stem = None
        for run_label in (match["run"].capitalize(), match["run"]):
            candidate = f'{match["systematic"]}_{match["detector"]}_{run_label}'
            if any(k.split(';')[0] == candidate for k in rf.keys()):
                stem = candidate
                break
        if stem is None:
            print(f'  no histogram found for {base_name}, skipping')
            continue
        hratio_key = stem
        hscaled_key = f'{stem}_scaled'
        hnominal_key = f'{stem}_nominal'
        # Flat systematics write the ratio alone, with no nominal/scaled pair.
        if not any(k.split(';')[0] == hnominal_key for k in rf.keys()):
            print(f'  {base_name}: ratio only (no nominal/scaled), skipping')
            continue
        data[base_name] = {
            'detector': match.group('detector'),
            'run': match.group('run'),
            'systematic': match.group('systematic'),
            'variable': match.group('variable'),
            'hratio': load_histogram(rf, hratio_key),
            'hscaled': load_histogram(rf, hscaled_key),
            'hnominal': load_histogram(rf, hnominal_key)
        }

print(f'loaded {len(data)} histogram sets')


In [ ]:
# Example dictionary content
"""
{'detector': 'sbnd',
 'run': 'run1',
 'systematic': 'ChargeScale',
 'variable': 'by_vis_energy',
 'hratio': (array([0.3 , 0.4 , 0.46, 0.52, 0.58, 0.64, 0.7 , 0.76, 0.82, 0.88, 0.94,
         1.  , 1.06, 1.13, 1.2 , 1.3 , 1.5 , 2.  ]),
  array([1.013947  , 1.00849784, 1.01093414, 1.00030074, 0.99809921,
         0.99794387, 0.99540754, 0.98985835, 0.98694982, 0.98553689,
         0.98358025, 0.98314515, 0.98172522, 0.96627365, 0.9789319 ,
         0.9570233 , 0.95779817])),
 'hscaled': (array([0.3 , 0.4 , 0.46, 0.52, 0.58, 0.64, 0.7 , 0.76, 0.82, 0.88, 0.94,
         1.  , 1.06, 1.13, 1.2 , 1.3 , 1.5 , 2.  ]),
  array([10905., 23142., 31990., 36587., 38332., 37372., 34463., 29769.,
         24957., 20238., 15934., 12016., 10368.,  7048.,  5994.,  4231.,
          1044.])),
 'hnominal': (array([0.3 , 0.4 , 0.46, 0.52, 0.58, 0.64, 0.7 , 0.76, 0.82, 0.88, 0.94,
         1.  , 1.06, 1.13, 1.2 , 1.3 , 1.5 , 2.  ]),
  array([10755., 22947., 31644., 36576., 38405., 37449., 34622., 30074.,
         25287., 20535., 16200., 12222., 10561.,  7294.,  6123.,  4421.,
          1090.]))}
"""

def plot_ratio(
    data: dict | list[dict],
    sigma: float | list[float] | None = None,
    xlabel: str = 'Visible Energy [GeV]',
    left_top_label: str = 'SBN Analysis-in-Progress',
    right_top_label: str = 'Detector Systematics'
) -> None:
    """
    Visualize the impact of detector systematics on the variable of
    interest with two plot elements:
    
    1. A top panel showing the nominal histogram and the systematic
    variation histogram(s) overlaid.

    2. A bottom panel showing the ratio of the systematic variation(s)
    to the nominal histogram.
    
    Optionally accepts a list of dictionaries to compare multiple
    systematics, in which case the first "nominal" histogram is used as
    the common CV to which all ratios are applied.

    Parameters
    ----------
    data : dict | list[dict]
        A dictionary or list of dictionaries containing the histogram
        data to plot. Each dictionary should have the following keys:
        - 'detector': str, the detector name (e.g., 'sbnd', 'icarus')
        - 'run': str, the run identifier (e.g., 'run1', 'run2')
        - 'systematic': str, the systematic variation name
        - 'variable': str, the variable name (e.g., 'by_vis_energy')
        - 'hratio': tuple of (bin_edges, bin_contents) for the ratio
        - 'hscaled': tuple of (bin_edges, bin_contents) for the scaled histogram
        - 'hnominal': tuple of (bin_edges, bin_contents) for the nominal histogram
    sigma : float | list[float] | None, optional
        An override for the strength of the pull term(s). If provided,
        it will be used to scale up or down the strength of the ratio
        by stretching it away from 1.0.
    xlabel : str, optional
        The label for the x-axis of the bottom panel.
    left_top_label : str, optional
        A label to place in the top left corner of the top panel.
    right_top_label : str, optional
        A label to place in the top right corner of the top panel.
    
    Returns
    -------
    None
    """
    if isinstance(data, dict):
        data = [data]

    if sigma is not None:
        if isinstance(sigma, (int, float)):
            sigma = [sigma] * len(data)
        elif len(sigma) != len(data):
            raise ValueError("Length of sigma must match length of data.")

        for d, s in zip(data, sigma):
            bin_edges_ratio, bin_contents_ratio = d['hratio']
            bin_contents_ratio = 1.0 + (bin_contents_ratio - 1.0) * (1/s)
            d['hratio'] = (bin_edges_ratio, bin_contents_ratio)

    # Create the figure and axes
    fig, (ax_top, ax_bottom) = plt.subplots(
        nrows=2,
        ncols=1,
        sharex=True,
        gridspec_kw={'height_ratios': [3, 1]},
        figsize=(8, 6)
    )

    # Plot the nominal histogram on the top panel
    bin_edges_nominal, bin_contents_nominal = data[0]['hnominal']
    ax_top.stairs(bin_contents_nominal, bin_edges_nominal, label='Nominal', color='black')

    colors = ['C0', 'C2', 'C3', 'C4', 'C5']

    # Plot the systematic variations on the top panel
    for d, color in zip(data, colors):
        bin_edges_scaled, _ = d['hscaled']
        bin_contents_scaled = data[0]['hnominal'][1] * d['hratio'][1]
        ax_top.stairs(bin_contents_scaled, bin_edges_scaled, label=d['systematic'], color=color)

    ax_top.set_ylabel('Events')
    ax_top.legend()

    # Plot the ratio on the bottom panel
    print(bin_edges_nominal)
    for d, color in zip(data, colors):
        bin_edges_ratio, bin_contents_ratio = d['hratio']
        ax_bottom.step(
            bin_edges_ratio,
            np.concatenate([bin_contents_ratio, [bin_contents_ratio[-1]]]),
            where='post',
            label=d['systematic'],
            color=color
        )

    ax_bottom.axhline(1.0, color='black', linestyle='--')
    ax_bottom.set_xlabel(xlabel)
    ax_bottom.set_ylabel('Ratio')
    ax_bottom.set_ylim(0.9, 1.1)

    # Scientific limits for y-axis
    ax_top.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

    # Mark axis
    mark_axis(ax_top, left_top_label, hadj=0.035)
    mark_axis(ax_top, right_top_label, alignment='right')

plot_ratio(
    [
        data['sbnd_run1_ChargeScale_by_vis_energy'],
        data['sbnd_run1_GainHigh_by_vis_energy'],
        data['sbnd_run1_GainLow_by_vis_energy'],
        data['sbnd_run1_WireModxThetaXW_by_vis_energy'],
        data['sbnd_run1_WireModYZ_by_vis_energy'],
    ],
    sigma=[1.0, 2.5, 2.5, 1.0, 1.0,],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{SBND}$ Run 1'
)

plot_ratio(
    [
        data['sbnd_run1_DENT_by_vis_energy'],
        data['sbnd_run1_SCE0x_by_vis_energy'],
        data['sbnd_run1_SCE2x_by_vis_energy'],
    ],
    sigma=[1.0, 1.0, 1.0,],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{SBND}$ Run 1'
)

In [ ]:
plot_ratio(
    [
        data['icarus_run2_ChargeScale_by_vis_energy'],
        data['icarus_run2_WireModxThetaXW_by_vis_energy'],
        data['icarus_run2_WireModYZ_by_vis_energy'],
        data['icarus_run2_SCE0x_by_vis_energy'],
        data['icarus_run2_SCE2x_by_vis_energy'],

    ],
    sigma=[1.0, 1.0, 1.0, 1.0, 1.0,],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{ICARUS}$ Run 2'
)

plot_ratio(
    [
        data['icarus_run4_ChargeScale_by_vis_energy'],
        data['icarus_run4_WireModxThetaXW_by_vis_energy'],
        data['icarus_run4_WireModYZ_by_vis_energy'],
        data['icarus_run4_SCE0x_by_vis_energy'],
        data['icarus_run4_SCE2x_by_vis_energy'],

    ],
    sigma=[1.0, 1.0, 1.0, 1.0, 1.0,],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{ICARUS}$ Run 4'
)

In [ ]:
# SBND detector systematics in reconstructed energy, including the new
# MattModYZ variation (sbnd_detvar_mattmod_yz_nosyst.root, one-sided like DENT).
# ChargeScale15 rather than ChargeScale, and Gain/SCE left out, matching what
# the coverage and contour XMLs actually enable for SBND.
plot_ratio(
    [
        data['sbnd_run1_ChargeScale15_by_vis_energy'],
        data['sbnd_run1_WireModxThetaXW_by_vis_energy'],
        data['sbnd_run1_WireModYZ_by_vis_energy'],
        data['sbnd_run1_MattModYZ_by_vis_energy'],
        data['sbnd_run1_DENT_by_vis_energy'],
    ],
    sigma=[1.0, 1.0, 1.0, 1.0, 1.0],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{SBND}$ Run 1'
)


In [ ]:
# The two YZ treatments side by side. They are alternative descriptions of the
# same wire-response effect, so this is the plot that says whether MattModYZ
# should REPLACE WireModYZ or be added alongside it: adding both would
# double-count if they describe the same thing.
#
# In reconstructed energy MattModYZ is the smaller of the two -- mean
# |ratio - 1| of 0.6% against 2.6% -- and is a monotonic tilt (a ~0.5% deficit
# below 0.7 GeV rising to an excess above 1 GeV) rather than WireModYZ's
# larger, flatter suppression. The top bin (1.5-2.0 GeV, 1585 CV events) is the
# least certain: the variation and CV samples share ~95% of their events, so
# the ratio's statistical error is correlation-suppressed but not zero.
plot_ratio(
    [
        data['sbnd_run1_WireModYZ_by_vis_energy'],
        data['sbnd_run1_MattModYZ_by_vis_energy'],
    ],
    sigma=[1.0, 1.0],
    left_top_label=r'$\bf{SBN}$ Analysis-in-Progress',
    right_top_label=r'$\bf{SBND}$ Run 1: YZ treatments'
)
